# LoRA GPT-2 Medium WebNLG Training + Evaluation on Google Colab

Mirrors the E2E and DART notebooks for the WebNLG 2017 Challenge data-to-text task. Hyperparameters follow Hu et al. 2021 Table 11 (WebNLG column), which is identical to E2E except length penalty 0.8. Everything else (rank `4`, alpha `32`, dropout `0.1`, AdamW LR `2e-4`, weight decay `0.01`, label smoothing `0.1`, 5 epochs, beam `10`) matches the E2E baseline.

Recommended runtime: `Runtime > Change runtime type > GPU` (T4 is fine for GPT-2 Medium).

## 1. Check GPU

In [ ]:
!nvidia-smi

import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 2. Clone Or Update The Repo

In [ ]:
from getpass import getpass
from pathlib import Path
import os
import subprocess

REPO_OWNER = "justinlxiang"
REPO_NAME = "CS4782-final-project"
BRANCH = "webnlg-experiment"
PROJECT_DIR = Path("/content") / REPO_NAME
WORK_DIR = PROJECT_DIR / "lora-gpt2-medium-e2e"

token = getpass("GitHub token, or press Enter for public clone: ")
repo_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
if token:
    repo_url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(PROJECT_DIR)], check=True)

os.chdir(WORK_DIR)
print("working directory:", Path.cwd())
!git log --oneline -3

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Mount Google Drive

Backups go under one WebNLG-specific run folder so they don't collide with E2E or DART runs:

```text
/content/drive/MyDrive/webnlg_lora_r4_alpha32/
```

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

DRIVE_RUN_DIR = Path('/content/drive/MyDrive/webnlg_lora_r4_alpha32')
LOCAL_RUN_DIR = Path('outputs/runs/webnlg_lora_r4_alpha32')
LOCAL_ADAPTER = LOCAL_RUN_DIR / 'checkpoints' / 'adapter_final.pt'
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)


def backup_to_run(relative_path, destination_name=None):
    source = Path(relative_path)
    if not source.exists():
        print('skip missing:', source)
        return None
    destination = DRIVE_RUN_DIR / (destination_name or source.name)
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
    print('backed up:', source, '->', destination)
    return destination

print('Drive run dir:', DRIVE_RUN_DIR)
print('Local run dir:', LOCAL_RUN_DIR)

## 5. Download WebNLG Dataset

Microsoft's LoRA repo hosts the WebNLG 2017 Challenge JSON splits at
`examples/NLG/data/webnlg_challenge_2017/`. Each example carries a
`modifiedtripleset` and one or more `lexicalisations`.

In [ ]:
!mkdir -p data/raw/webnlg
!curl -L -o data/raw/webnlg/train.json https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/webnlg_challenge_2017/train.json
!curl -L -o data/raw/webnlg/dev.json   https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/webnlg_challenge_2017/dev.json
!curl -L -o data/raw/webnlg/test.json  https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/webnlg_challenge_2017/test.json
!ls -lh data/raw/webnlg/

## 6. Linearize WebNLG Triples → JSONL

`prepare_webnlg_raw.py` flattens each `(tripleset, good-lexicalisation)` pair into one `{context, completion}` row using the `<H> s <R> p <T> o` convention shared with DART. It also writes a multi-reference `references_test.txt` for the official evaluator.

In [ ]:
!python scripts/prepare_webnlg_raw.py --config configs/webnlg_gpt2_medium_lora.yaml

import json
from pathlib import Path
sample = json.loads(Path('data/raw/webnlg/train.jsonl').read_text().splitlines()[0])
print('context:   ', sample['context'][:200])
print('completion:', sample['completion'])
!head -5 data/processed/webnlg_gpt2/references_test.txt

## 7. Tokenize

`prepare_e2e.py` is dataset-agnostic (reads any `{context, completion}` JSONL) so it works for WebNLG unchanged.

In [ ]:
!python scripts/prepare_e2e.py --config configs/webnlg_gpt2_medium_lora.yaml

import json
from pathlib import Path
example = json.loads(Path('data/processed/webnlg_gpt2/train.jsonl').read_text().splitlines()[0])
print('prompt:', example['prompt'][:200])
print('first input ids:', example['input_ids'][:20])
print('first labels:', example['labels'][:20])
print('prompt length:', example['prompt_length'])

## 8. Run Tests And Dry Run

In [ ]:
!python -m pytest
!python scripts/count_params.py --config configs/webnlg_gpt2_medium_lora.yaml
!python scripts/train.py --config configs/webnlg_gpt2_medium_lora.yaml --dry-run --device cuda --dry-run-forward-pass

## 9. Optional: Short Smoke Training

In [ ]:
RUN_SMOKE_TRAIN = False

if RUN_SMOKE_TRAIN:
    !python scripts/train.py --config configs/webnlg_gpt2_medium_lora.yaml --smoke-train --device cuda --dry-run-max-examples 80 --dry-run-batch-size 8 --smoke-max-steps 10

## 10. Full Training

WebNLG config: rank `4`, alpha `32`, dropout `0.1`, AdamW LR `2e-4`, weight decay `0.01`, label smoothing `0.1`, 5 epochs, warmup `500`, linear schedule. End-of-epoch validation loss/perplexity is appended to `metrics.jsonl`.

In [ ]:
!python scripts/train.py --config configs/webnlg_gpt2_medium_lora.yaml --train --device cuda

## 11. Inspect Training Logs

In [ ]:
import json
from pathlib import Path

metrics_path = Path('outputs/runs/webnlg_lora_r4_alpha32/metrics.jsonl')
records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
validation = [record for record in records if record.get('type') == 'validation']
print('total metric records:', len(records))
print('validation records:')
for record in validation:
    print(record)

!ls -lh outputs/runs/webnlg_lora_r4_alpha32/checkpoints | tail

## 12. Back Up Training Outputs To Drive

In [ ]:
backup_to_run(LOCAL_RUN_DIR, destination_name='.')
backup_to_run('configs/webnlg_gpt2_medium_lora.yaml', destination_name='config_source.yaml')
backup_to_run('colab_train_lora_webnlg.ipynb')

print('Training backup run folder:', DRIVE_RUN_DIR)
!du -sh /content/drive/MyDrive/webnlg_lora_r4_alpha32
!find /content/drive/MyDrive/webnlg_lora_r4_alpha32 -maxdepth 2 -type f | sort | tail -20

## 13. Generate Test Predictions

In [ ]:
BATCH_SIZE = 16

!TOKENIZERS_PARALLELISM=false TRANSFORMERS_VERBOSITY=error python scripts/generate.py --config configs/webnlg_gpt2_medium_lora.yaml --split test --adapter "$LOCAL_ADAPTER" --batch-size "$BATCH_SIZE"

!ls -lh outputs/runs/webnlg_lora_r4_alpha32/generations_test.txt
!wc -l outputs/runs/webnlg_lora_r4_alpha32/generations_test.txt

## 14a. Quick Metrics (sacreBLEU + ROUGE-L)

Same evaluator as E2E/DART; multi-reference grouping uses `references_test.txt` written in step 6.

In [ ]:
!python scripts/evaluate.py --config configs/webnlg_gpt2_medium_lora.yaml
!cat outputs/runs/webnlg_lora_r4_alpha32/generations_test.metrics.json

## 14b. Optional: Official WebNLG Metrics (BLEU / METEOR / TER)

The WebNLG and DART evaluators both reuse the E2E `measure_scores.py` interface. Clone the DART repo and point at the bundled scoring code.

In [ ]:
!mkdir -p external
![ -d external/webnlg-metrics/.git ] || git clone https://github.com/Yale-LILY/dart.git external/webnlg-metrics

REF_FILE="data/processed/webnlg_gpt2/references_test.txt"
PRED_FILE="outputs/runs/webnlg_lora_r4_alpha32/generations_test.txt"
OUT_FILE="outputs/runs/webnlg_lora_r4_alpha32/generations_test.official_webnlg_metrics.txt"

!python external/webnlg-metrics/evaluation/e2e-metrics/measure_scores.py "$REF_FILE" "$PRED_FILE" -p 2>&1 | tee "$OUT_FILE" 

## 15. Create Figures

In [ ]:
!python scripts/make_figures.py --config configs/webnlg_gpt2_medium_lora.yaml --figures-dir outputs/runs/webnlg_lora_r4_alpha32/figures

!ls -lh outputs/runs/webnlg_lora_r4_alpha32/figures
!cat outputs/runs/webnlg_lora_r4_alpha32/figures/summary.json

## 16. Back Up Evaluation Outputs To Drive

In [ ]:
backup_to_run(LOCAL_RUN_DIR, destination_name='.')
backup_to_run('configs/webnlg_gpt2_medium_lora.yaml', destination_name='config_source.yaml')
backup_to_run('colab_train_lora_webnlg.ipynb')

print('Backed up complete run to:', DRIVE_RUN_DIR)
!find /content/drive/MyDrive/webnlg_lora_r4_alpha32 -maxdepth 3 -type f | sort | tail -60